# Trực quan hóa phân bố lớp: có SupCon vs không có SupCon

So sánh cosine giữa hai vector CLS của cặp clone/non-clone, AUC khi chỉ dùng cosine, t-SNE trên hA.
Chỉ phân tích — không tune ngưỡng trên test.


In [ ]:
import base64, io, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_viz'
CLEAN.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f'\n$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with (WORK / 'live_progress.log').open('a', encoding='utf-8') as lf:
        for line in iter(p.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line); lf.flush()
    p.stdout.close()
    rc = p.wait()
    if rc != 0: raise RuntimeError(f'Fail rc={rc}: {cmd}')


In [ ]:
payload = '...'
repo_root = WORK / 'project'
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as tar:
    tar.extractall(repo_root)
os.chdir(repo_root)
print('Repo:', os.getcwd(), flush=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available(): raise SystemError('Chua bat GPU!')
sh('pip install -q -U transformers datasets scikit-learn scipy pyarrow matplotlib')


In [ ]:
data_raw = repo_root / 'data/raw/bigclonebench'
data_raw.mkdir(parents=True, exist_ok=True)
ckpts_dir = repo_root / 'results/ckpts'
ckpts_dir.mkdir(parents=True, exist_ok=True)

files = {
    'bcb_test.parquet': data_raw / 'bcb_test.parquet',
    'm2_diffmul_best.pt': ckpts_dir / 'm2_diffmul_best.pt',
    'm2_b5b6_best.pt': ckpts_dir / 'm2_b5b6_best.pt',
}
for name, dest in files.items():
    cands = list(Path('/kaggle/input').rglob(name))
    if cands and not dest.exists(): sh(f'cp "{cands[0]}" "{dest}"')
    print(name, dest.exists(), flush=True)
    assert dest.exists(), f'Thieu {name}'


In [ ]:
print('=' * 70)
print('>>> BAT DAU SUP- CON EMBEDDING ANALYSIS <<<')
print('=' * 70)
t0 = time.perf_counter()
sh('python scripts/analyze_supcon_embeddings.py '
   '--no-supcon results/ckpts/m2_diffmul_best.pt '
   '--with-supcon results/ckpts/m2_b5b6_best.pt '
   '--pairs-file data/raw/bigclonebench/bcb_test.parquet '
   '--out-dir results/supcon_embedding --batch-size 32')
print(f'>>> XONG TRONG {(time.perf_counter()-t0)/60:.1f} PHUT', flush=True)
print((repo_root / 'results/supcon_embedding/report.json').read_text(encoding='utf-8'), flush=True)


In [ ]:
out_dir = repo_root / 'results/supcon_embedding'
for p in out_dir.rglob('*'):
    if p.is_file() and p.suffix in ('.json', '.png', '.npy'):
        name = 'viz_' + p.name
        shutil.copy2(p, CLEAN / name)
        shutil.copy2(p, WORK / name)
        print('  [SAVED]', name, flush=True)
log = WORK / 'live_progress.log'
if log.exists():
    shutil.copy2(log, CLEAN / 'viz_live_progress.log')
    shutil.copy2(log, WORK / 'viz_live_progress.log')
with tarfile.open(WORK / 'supcon_viz_bundle.tar.gz', 'w:gz') as tar:
    for f in CLEAN.glob('*'): tar.add(f, arcname=f.name)
os.chdir(WORK)
for junk in ('project', 'data', 'clean_viz'):
    shutil.rmtree(WORK / junk, ignore_errors=True)
print('>>> XONG 100%!', flush=True)